In [ ]:
import pandas as pd
import json

# 1) CSV 파일 경로 지정
input_csv = "/content/drive/MyDrive/캡스톤디자인/subset_8075_11949_ko.csv"
output_json = "/content/drive/MyDrive/캡스톤디자인/subset_8075_11949_ko.json"

# 2) CSV 불러오기
df = pd.read_csv(input_csv)
if "rn" in df.columns:
    df = df.drop(columns=["rn"])

# 3) 열 이름 매핑 (ko → 영문)
df = df.rename(columns={
    "Question_ko": "Question",
    "Complex_CoT_ko": "Complex_CoT",
    "Response_ko": "Response"
})

# 4) JSON 변환
records = df.to_dict(orient="records")

# 5) 파일 저장 (UTF-8, indent 보기 좋게)
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"JSON 변환 완료: {output_json}")


JSON 변환 완료: /content/drive/MyDrive/캡스톤디자인/subset_8075_11949_ko.json


/tmp/ipython-input-2473906211.py:21: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  records = df.to_dict(orient="records")


In [ ]:
import json
import glob

# 1) 합칠 JSON 파일들 경로 지정 (예: 5개 파일이 같은 폴더에 있을 때)
input_files = [
    "/content/drive/MyDrive/캡스톤디자인/combined_translated_0000_3999.json",
    "/content/drive/MyDrive/캡스톤디자인/medical_o1_sft_rows_11950-15824_ko.json",
    "/content/drive/MyDrive/캡스톤디자인/medical_o1_sft_trans_15825_19700.json",
    "/content/drive/MyDrive/캡스톤디자인/subset_8075_11949_ko.json",
    "/content/drive/MyDrive/캡스톤디자인/translated_4200_8074.json"
]

output_file = "/content/drive/MyDrive/캡스톤디자인/medical_translated_0000_19700.json"

# 2) 파일들 읽어서 리스트 합치기
merged_data = []
for file in input_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)   # 각 파일은 리스트임
        merged_data.extend(data)

# 3) 합친 결과 저장
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=2)

print(f"총 {len(merged_data)}개의 항목을 {output_file}에 저장했습니다.")


총 19158개의 항목을 /content/drive/MyDrive/캡스톤디자인/medical_translated_0000_19700.json에 저장했습니다.


In [1]:
%%capture
!pip install -q unsloth # install unsloth. automatically resolve dependencies
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# **검증하기**

In [7]:
from datasets import Dataset
import json
import re

# 1. 두 JSON 파일 경로
json_paths = [
    "/content/drive/MyDrive/캡스톤디자인/validation_translated_4001_4100.json",
    "/content/drive/MyDrive/캡스톤디자인/validation_translated_4101_4200.json",
]

# 2. 한자(중국어)가 포함되었는지 확인하는 함수
def contains_chinese(text):
    return bool(re.search(r'[\u4e00-\u9fff]', text))

def remove_cot(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

# 3. 병합 및 클렌징
cleaned_data = []
skipped_chinese = 0
skipped_incomplete = 0

for path in json_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:
        if all(k in item for k in ["Question_ko", "Complex_CoT_ko", "Response_ko"]):
            q, c, r = item["Question_ko"], item["Complex_CoT_ko"], item["Response_ko"]
        elif all(k in item for k in ["Question", "Complex_CoT", "Response"]):
            q, c, r = item["Question"], item["Complex_CoT"], item["Response"]
        else:
            skipped_incomplete += 1
            continue

        if contains_chinese(q) or contains_chinese(c) or contains_chinese(r):
            skipped_chinese += 1
            continue

        r_clean = remove_cot(r)

        cleaned_data.append({
            "Question": q.strip(),
            "Response": r_clean.strip(),
        })

# 4. 통계 출력
print(f"✅ 최종 유효한 항목 수: {len(cleaned_data)}")
print(f"🚫 제거된 중국어 포함 항목 수: {skipped_chinese}")
print(f"🚫 키 누락 등으로 제외된 항목 수: {skipped_incomplete}")

# 5. 🤗 Hugging Face Dataset 변환
dataset = Dataset.from_list(cleaned_data)

# 필요 시 확인
print(dataset[0])


✅ 최종 유효한 항목 수: 198
🚫 제거된 중국어 포함 항목 수: 2
🚫 키 누락 등으로 제외된 항목 수: 0
{'Question': '2살짜리 아이가 발열과 구토, 그리고 경부 강직 증상을 보이며 응급실에 왔습니다. 뇌척수액 검사 결과 백혈구 수는 2000/µL 이고 단백질 수치는 100 mg/dL 입니다. 그람염색 결과 그람음성 간균이 나타났습니다. 세균 배양은 초콜릿 한천 배지에서는 자라지만 혈액 한천 배지에서는 자라지 않았습니다. 이 아이의 증상의 가장 가능성이 높은 원인균은 무엇입니까?', 'Response': '이 아이의 증상을 유발한 가장 가능성 높은 원인균은 *Haemophilus influenzae*입니다. 임상 증상, 뇌척수액 검사 결과, Gram stain 결과, 그리고 초콜릿 한천 배지에서만 자라는 특이한 성장 양상 모두 *Haemophilus influenzae*에 의한 감염과 일치합니다. 이 세균은 어린아이들에게서 *bacterial meningitis*를 일으키는 것으로 알려져 있으며, 초콜릿 한천 배지에 포함된 특정 성장 인자를 필요로 하기 때문에 혈액 한천 배지에서는 자라지 않습니다.'}


In [1]:
!pip install bitsandbytes==0.43.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 16.2 MB/s eta 0:00:00


In [1]:
!pip install -U bitsandbytes

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 모델 이름 (허깅페이스 허브 경로)
model_name = "nanyaas/deepseek-r1-medicalQA-Qwen_7B_dev"

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                # 4bit 로드
    bnb_4bit_use_double_quant=True,   # double quantization 사용
    bnb_4bit_quant_type="nf4",        # 양자화 타입: NormalFloat4
    bnb_4bit_compute_dtype=torch.bfloat16,  # GPU 지원 안 하면 torch.float16 으로 변경
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # GPU 자동 할당
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.52G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/80.8M [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
    (layers): ModuleList(
      (0-3): 4 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3584, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=3584, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=3584, out_features=512, bias=True)
            (lora_dropout): ModuleDict(
              (default): Iden

In [3]:
def make_prompt(question):
    return f"""
아래는 특정 작업에 대한 지침과 함께, 추가적인 맥락을 제공하는 입력이 주어져 있습니다.
요청에 적절히 응답하는 답변을 작성하세요.
답변을 작성하기 전에 질문을 신중히 분석하고, 단계적인 사고 과정을 통해 논리적이고 정확한 결론에 도달하세요.

### 지침:
당신은 임상 추론, 진단, 치료 계획에 대해 고도의 전문 지식을 갖춘 의료 전문가입니다.
다음 의학 질문에 대해 **한국어로만** 답변해 주세요.

### 질문:
{question}

### 응답:
<think>
"""


In [4]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00


In [5]:
!pip install bert-score sentence-transformers


데이터셋 100개

In [9]:
from datasets import Dataset
import json
import re

# 1. 두 JSON 파일 경로
json_paths = [
    "/content/drive/MyDrive/캡스톤디자인/validation_translated_4001_4100.json",
    "/content/drive/MyDrive/캡스톤디자인/validation_translated_4101_4200.json",
]

def contains_chinese(text):
    return bool(re.search(r'[\u4e00-\u9fff]', text))

def remove_cot(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

cleaned_data, skipped_chinese, skipped_incomplete = [], 0, 0
for path in json_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        if all(k in item for k in ["Question_ko", "Complex_CoT_ko", "Response_ko"]):
            q, c, r = item["Question_ko"], item["Complex_CoT_ko"], item["Response_ko"]
        elif all(k in item for k in ["Question", "Complex_CoT", "Response"]):
            q, c, r = item["Question"], item["Complex_CoT"], item["Response"]
        else:
            skipped_incomplete += 1
            continue

        if contains_chinese(q) or contains_chinese(c) or contains_chinese(r):
            skipped_chinese += 1
            continue

        r_clean = remove_cot(r)
        cleaned_data.append({"Question": q.strip(), "Response": r_clean.strip()})

print(f"✅ 최종 유효한 항목 수: {len(cleaned_data)}")
print(f"🚫 제거된 중국어 포함 항목 수: {skipped_chinese}")
print(f"🚫 키 누락 등으로 제외된 항목 수: {skipped_incomplete}")

dataset = Dataset.from_list(cleaned_data)

subset = dataset.select(range(min(30, len(dataset))))
print(subset[0])  # 확인

from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import pandas as pd
import torch

sbert_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

predictions, references, questions = [], [], []

for item in tqdm(subset, total=len(subset), desc="Evaluating (30 samples)"):
    q = item["Question"]
    gt = item["Response"]

    prompt = make_prompt(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(**inputs, max_new_tokens=600)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred_full = response.split("### 응답:")[-1].strip()
    pred = pred_full.split("</think>")[-1].strip() if "</think>" in pred_full else pred_full

    questions.append(q)
    predictions.append(pred)
    references.append(gt)

P, R, F1 = bert_score(predictions, references, lang="ko", verbose=True)
bert_f1_scores = [round(f.item(), 4) for f in F1]
avg_bert_f1 = F1.mean().item() * 100

cosine_scores = []
for pred, ref in zip(predictions, references):
    emb1 = sbert_model.encode(pred, convert_to_tensor=True)
    emb2 = sbert_model.encode(ref, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb1, emb2).item()
    cosine_scores.append(round(sim, 4))
avg_cosine = sum(cosine_scores) / len(cosine_scores) * 100

df = pd.DataFrame({
    "Question": questions,
    "Prediction": predictions,
    "Reference": references,
    "BERTScore_F1": bert_f1_scores,
    "CosineSimilarity": cosine_scores
})

print(f"✅ 평균 의미 유사도 (BERTScore F1): {avg_bert_f1:.2f}%")
print(f"✅ 평균 코사인 유사도: {avg_cosine:.2f}%")

df.to_csv("evaluation_results_30.csv", index=False)


✅ 최종 유효한 항목 수: 198
🚫 제거된 중국어 포함 항목 수: 2
🚫 키 누락 등으로 제외된 항목 수: 0
{'Question': '2살짜리 아이가 발열과 구토, 그리고 경부 강직 증상을 보이며 응급실에 왔습니다. 뇌척수액 검사 결과 백혈구 수는 2000/µL 이고 단백질 수치는 100 mg/dL 입니다. 그람염색 결과 그람음성 간균이 나타났습니다. 세균 배양은 초콜릿 한천 배지에서는 자라지만 혈액 한천 배지에서는 자라지 않았습니다. 이 아이의 증상의 가장 가능성이 높은 원인균은 무엇입니까?', 'Response': '이 아이의 증상을 유발한 가장 가능성 높은 원인균은 *Haemophilus influenzae*입니다. 임상 증상, 뇌척수액 검사 결과, Gram stain 결과, 그리고 초콜릿 한천 배지에서만 자라는 특이한 성장 양상 모두 *Haemophilus influenzae*에 의한 감염과 일치합니다. 이 세균은 어린아이들에게서 *bacterial meningitis*를 일으키는 것으로 알려져 있으며, 초콜릿 한천 배지에 포함된 특정 성장 인자를 필요로 하기 때문에 혈액 한천 배지에서는 자라지 않습니다.'}


Evaluating (30 samples): 100%|██████████| 30/30 [31:36<00:00, 63.21s/it]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.91 seconds, 32.99 sentences/sec
✅ 평균 의미 유사도 (BERTScore F1): 70.55%
✅ 평균 코사인 유사도: 71.42%


In [3]:

from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import pandas as pd

# ✅ 1. 문장 임베딩 모델 로드 (코사인 유사도용)
sbert_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# ✅ 2. 평가용 데이터 준비
predictions = []
references = []
questions = []

for item in tqdm(dataset):
    q = item["Question"]
    gt = item["Response"]

    # 프롬프트 생성
    prompt = make_prompt(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=600)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred_full = response.split("### 응답:")[-1].strip()
    pred = pred_full.split("</think>")[-1].strip() if "</think>" in pred_full else pred_full

    # 결과 저장
    questions.append(q)
    predictions.append(pred)
    references.append(gt)

# ✅ 3. BERTScore 계산 (의미 기반 정밀 유사도)
P, R, F1 = bert_score(predictions, references, lang="ko", verbose=True)
bert_f1_scores = [round(f.item(), 4) for f in F1]
avg_bert_f1 = F1.mean().item() * 100

# ✅ 4. 코사인 유사도 계산 (문장 임베딩 유사도)
cosine_scores = []
for pred, ref in zip(predictions, references):
    emb1 = sbert_model.encode(pred, convert_to_tensor=True)
    emb2 = sbert_model.encode(ref, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb1, emb2).item()
    cosine_scores.append(round(sim, 4))
avg_cosine = sum(cosine_scores) / len(cosine_scores) * 100

# ✅ 5. 결과 정리
df = pd.DataFrame({
    "Question": questions,
    "Prediction": predictions,
    "Reference": references,
    "BERTScore_F1": bert_f1_scores,
    "CosineSimilarity": cosine_scores
})

# ✅ 6. 요약 출력
print(f"✅ 평균 의미 유사도 (BERTScore F1): {avg_bert_f1:.2f}%")
print(f"✅ 평균 코사인 유사도: {avg_cosine:.2f}%")

# ✅ 7. (선택) CSV로 저장
df.to_csv("evaluation_results.csv", index=False)


ModuleNotFoundError: No module named 'bert_score'

데이터셋 200개

In [ ]:

from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import pandas as pd

# ✅ 1. 문장 임베딩 모델 로드 (코사인 유사도용)
sbert_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# ✅ 2. 평가용 데이터 준비
predictions = []
references = []
questions = []

for item in tqdm(dataset):
    q = item["Question"]
    gt = item["Response"]

    # 프롬프트 생성
    prompt = make_prompt(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=600)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred_full = response.split("### 응답:")[-1].strip()
    pred = pred_full.split("</think>")[-1].strip() if "</think>" in pred_full else pred_full

    # 결과 저장
    questions.append(q)
    predictions.append(pred)
    references.append(gt)

# ✅ 3. BERTScore 계산 (의미 기반 정밀 유사도)
P, R, F1 = bert_score(predictions, references, lang="ko", verbose=True)
bert_f1_scores = [round(f.item(), 4) for f in F1]
avg_bert_f1 = F1.mean().item() * 100

# ✅ 4. 코사인 유사도 계산 (문장 임베딩 유사도)
cosine_scores = []
for pred, ref in zip(predictions, references):
    emb1 = sbert_model.encode(pred, convert_to_tensor=True)
    emb2 = sbert_model.encode(ref, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb1, emb2).item()
    cosine_scores.append(round(sim, 4))
avg_cosine = sum(cosine_scores) / len(cosine_scores) * 100

# ✅ 5. 결과 정리
df = pd.DataFrame({
    "Question": questions,
    "Prediction": predictions,
    "Reference": references,
    "BERTScore_F1": bert_f1_scores,
    "CosineSimilarity": cosine_scores
})

# ✅ 6. 요약 출력
print(f"✅ 평균 의미 유사도 (BERTScore F1): {avg_bert_f1:.2f}%")
print(f"✅ 평균 코사인 유사도: {avg_cosine:.2f}%")

# ✅ 7. (선택) CSV로 저장
df.to_csv("evaluation_results_200.csv", index=False)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


 58%|█████▊    | 114/198 [2:02:06<1:28:29, 63.20s/it]